# 52 구조화 단독·비구조화 단독·순차·병렬 비교

개발 질문 40개, 51번의 고정된 1회차 조건을 사용한다. S_ONLY는 청킹과 무관하게 1회씩, U/SEQ/PAR는 두 운영 프로필에서 비교한다. 답변 280개 조건은 서로 새 호출 ID를 사용한다.

SEQ는 구조화 결과 카드 범위에서 원질문으로 비구조화 검색을 한 번만 수행한다. PAR는 DB 사실과 독립 U의 후보를 그대로 합치고 원질문으로 BGE 재정렬한다. 추가 전체 U를 SEQ에 섞지 않는다. 모든 방식에 원래 발급사·상품 범위를 동일하게 적용한다. 근거는 자르지 않는다. 공통 코드와 회귀 검사는 scripts/52_retrieval_modes.py 및 52_test_retrieval_modes.py에 둔다. 보고서 생성 코드는 만들지 않는다.

In [1]:
from pathlib import Path
import importlib.util,sys,json,os,time,copy,resource
ROOT=Path.cwd()
if not (ROOT/'notebooks').exists():ROOT=ROOT.parent
sys.dont_write_bytecode=True
spec=importlib.util.spec_from_file_location('modes52',ROOT/'notebooks/scripts/52_retrieval_modes.py')
M=importlib.util.module_from_spec(spec);sys.modules[spec.name]=M;spec.loader.exec_module(M)
X=M.X;P=M.P;Q=M.Q
recovery=X.load('budget_recovery52','51_budget_recovery.py');recovery.install(X)
PATH=P.F.STAGES[52];PATH.mkdir(exist_ok=True);P.F.initialize(52)
assert P.F.read_json(P.F.STAGES[51]/'completion_status.json')['stage52_allowed']
queries=P.F.read_json(P.F.STAGES[50]/'questions.json')['queries']
plans=P.F.read_json(P.F.STAGES[51]/'repeat1_plans.json')
by_query={r['parser']['query_id']:r for r in plans['records']}
assert len(by_query)==40 and all(r['parser']['repeat']==1 for r in by_query.values())
annotations=P.F.read_json(PATH/'evaluation_annotations.json')
identities=[f"{q['query_id']}:S_ONLY:r1" for q in queries]+[f"{q['query_id']}:{pf}:{mode}:r1" for q in queries for pf in X.PROFILES for mode in ['U','SEQ','PAR']]
conditional=[f"{q['query_id']}:{pf}:{mode}:r1" for q in queries for pf in X.PROFILES for mode in ['LLM_SELECTOR','PAR_LLM']]
previous=P.F.read_json(P.F.STAGES[51]/'execution_manifest.json')
sources={**previous['source_sha256'],'notebooks/scripts/52_retrieval_modes.py':P.sha((ROOT/'notebooks/scripts/52_retrieval_modes.py').read_bytes()),'notebooks/scripts/51_budget_recovery.py':P.sha((ROOT/'notebooks/scripts/51_budget_recovery.py').read_bytes())}
manifest=X.make_manifest(52,ROOT/'notebooks/52_four_retrieval_modes.ipynb',queries,dict(parser=[],answer=identities+conditional),
 maximum_answer=880,maximum_embedding=2,stage_cap='2',extra=dict(source_sha256=sources,split='development',
 logical_answer_conditions=280,conditional_selector_conditions=80,conditional_answer_conditions=80,
 repeat1_plans_sha256=P.sha(plans),annotations_sha256=P.sha(annotations),
 pool='all matching DB cards + independent U candidates; no SQL first20 cut',
 common_unit='whole serialized fact or unchanged operating U evidence chunk',
 card_score='max unit score; tie by card key; disclose unit counts',
 conditional_trigger='source-adjudicated valid card in fixed union but no valid card in BGE top3, ranking cause only'))
X.frozen(PATH/'questions.json',dict(queries=queries,split='development'))
X.frozen(PATH/'frozen_annotations.json',annotations)
ALLOW_NETWORK=os.getenv('PAID_RETEST_ALLOW_NETWORK')=='1'
print('답변 조건',len(identities),'조건부 대안',len(conditional),'누적비용',X.ledger())

답변 조건 280 조건부 대안 160 누적비용 {'record_count': 441, 'known_priced_records': 440, 'unknown_cost_records': 1, 'duplicate_saved_responses': 0, 'known_cost_usd': '0.272201310', 'tokens': {'input_tokens': 1867285, 'output_tokens': 301488, 'ordinary_input_tokens': 121566, 'cached_input_tokens': 948759, 'cache_write_tokens': 796960, 'embedding_tokens': 9656}, 'billed_cost_usd': None}


In [ ]:
class BudgetRuntime(X.Runtime):
 def embeddings(self,texts,*,allow_network=False):
  X.guard_manifest(self.manifest)
  start=time.perf_counter();target=self.path/'api_cache/embeddings'
  keys=[P.sha(dict(text=t,model='text-embedding-3-small',dimension=1536)) for t in texts]
  current={}
  batches=[P.F.read_json(p) for p in (self.path/'api_cache').glob('embedding_batch_*.json')]
  for text,key in zip(texts,keys,strict=True):
   path=target/(key+'.npy')
   if not path.exists():continue
   vector=self.np.load(path,allow_pickle=False)
   proof=None
   for batch in batches:
    request=batch.get('request',{})
    if text not in request.get('input',[]):continue
    assert batch['status']=='ok' and request['model']=='text-embedding-3-small' and request['dimensions']==1536
    item=next(r for r in batch['raw_response']['data'] if r['index']==request['input'].index(text))
    proof=self.np.asarray(item['embedding'],dtype=self.np.float32)
   if proof is None or not self.np.array_equal(vector,proof):raise RuntimeError('Current question embedding lacks unchanged raw provider proof')
   current[text]=vector
  remaining=list(dict.fromkeys(t for t in texts if t not in current))
  original_ledger=P.ledger
  try:
   P.ledger=X.budget_upper_ledger
   if remaining:current.update(zip(remaining,super().embeddings(remaining,allow_network=allow_network),strict=True))
  finally:P.ledger=original_ledger
  vectors=self.np.asarray([current[t] for t in texts],dtype=self.np.float32)
  assert vectors.shape==(len(texts),1536) and self.np.isfinite(vectors).all()
  self.last_embedding_seconds=time.perf_counter()-start
  X.frozen(self.path/'question_embedding_contract.json',dict(texts_sha256=P.sha(texts),model='text-embedding-3-small',dimensions=1536,matrix_sha256=P.sha(vectors.tobytes())))
  return vectors
runtime=BudgetRuntime(52)
started=time.perf_counter();cpu_started=time.process_time()
vectors=runtime.embeddings([q['question'] for q in queries],allow_network=ALLOW_NETWORK)
qvectors=dict(zip([q['query_id'] for q in queries],vectors,strict=True))
for pf in X.PROFILES:runtime.profile(pf)
retrievals=[];search_cache={}
for qi,q in enumerate(queries):
 record=by_query[q['query_id']];route=record['routing'];original=record['original_scope']
 identity=f"{q['query_id']}:S_ONLY:r1";target=PATH/'retrieval'/(identity+'.json')
 new=not target.exists()
 if not new: row=P.F.read_json(target)
 else:
  result=M.rank_units(runtime,q['question'],M.structured_units(route))
  if original['clarify']:result['clarify']=True
  M.assert_scope(result,original['card_keys'])
  row=dict(identity=identity,query_id=q['query_id'],profile='common',mode='S_ONLY',repeat=1,result=result,route=route,
   manifest_sha256=P.sha(manifest),original_scope=original,reuse=None)
 M.bind_retrieval(52,row,question=q,profile='common',mode='S_ONLY',parser_record=record,manifest=manifest,new=new)
 if new:P.F.write_json(target,row)
 retrievals.append(row)
 for pf in X.PROFILES:
  scopes={'U':original['card_keys'],'SEQ':route['scope']}
  results={}
  for mode in ['U','SEQ']:
   identity=f"{q['query_id']}:{pf}:{mode}:r1";target=PATH/'retrieval'/(identity+'.json')
   new=not target.exists()
   if not new:row=P.F.read_json(target)
   else:
    key=P.sha(dict(question=q['question'],profile=pf,scope=scopes[mode]));start=time.perf_counter();reuse=None
    if original['clarify']:result=dict(cards=[],evidence=[],trace={},clarify=True)
    elif key in search_cache:result=copy.deepcopy(search_cache[key]);reuse='same question/profile/scope; original search timing retained'
    else:result=runtime.search(q['question'],qvectors[q['query_id']],pf,scopes[mode])
    M.assert_scope(result,scopes[mode]);M.assert_scope(result,original['card_keys'])
    row=dict(identity=identity,query_id=q['query_id'],profile=pf,mode=mode,repeat=1,result=result,route=route if mode=='SEQ' else None,
     manifest_sha256=P.sha(manifest),original_scope=original,reuse=reuse,observed_retrieval_wall_seconds=time.perf_counter()-start)
   M.bind_retrieval(52,row,question=q,profile=pf,mode=mode,parser_record=record,manifest=manifest,new=new)
   if new:P.F.write_json(target,row)
   results[mode]=row['result'];search_cache[P.sha(dict(question=q['question'],profile=pf,scope=scopes[mode]))]=row['result'];retrievals.append(row)
  identity=f"{q['query_id']}:{pf}:PAR:r1";target=PATH/'retrieval'/(identity+'.json')
  new=not target.exists()
  if not new:row=P.F.read_json(target)
  else:
   pool=[] if original['clarify'] else M.union_units(M.structured_units(route),M.unstructured_units(runtime,results['U'],pf))
   result=M.rank_units(runtime,q['question'],pool)
   if original['clarify']:result['clarify']=True
   M.assert_scope(result,original['card_keys'])
   row=dict(identity=identity,query_id=q['query_id'],profile=pf,mode='PAR',repeat=1,result=result,route=route,
    manifest_sha256=P.sha(manifest),original_scope=original,reuse=None,independent_U_sha256=P.sha(results['U']))
  M.bind_retrieval(52,row,question=q,profile=pf,mode='PAR',parser_record=record,manifest=manifest,independent_u=results['U'],new=new)
  if new:P.F.write_json(target,row)
  retrievals.append(row)
 for r in retrievals[-7:]:assert r['manifest_sha256']==P.sha(manifest)
 P.F.write_json(PATH/'progress.json',dict(phase='retrieval',questions_completed=qi+1,logical_conditions=len(retrievals),ledger=X.ledger()))
 print('검색',q['query_id'],len(retrievals),flush=True)
P.F.write_json(PATH/'retrieval_results.json',dict(records=retrievals,resources=runtime.resources(),embedding_wall_seconds=runtime.last_embedding_seconds))
print('검색 완료',len(retrievals),'API 답변 생성 전')

검색 H01 7
검색 H02 14
검색 H03 21
검색 H04 28
검색 H05 35
검색 H06 42
검색 H07 49
검색 H08 56
검색 H09 63
검색 H10 70
검색 H11 77
검색 H12 84
검색 H13 91
검색 H14 98
검색 H15 105
검색 H16 112
검색 H17 119
검색 H18 126
검색 H19 133
검색 H20 140
검색 H21 147
검색 H22 154
검색 H23 161
검색 H24 168
검색 H25 175
검색 H26 182
검색 H27 189
검색 H28 196
검색 H29 203
검색 H30 210
검색 H31 217
검색 H32 224
`torch_dtype` is deprecated! Use `dtype` instead!
검색 D33 231


In [ ]:
answers=[]
# 질문별 방식 순서를 번갈아 배치한다. 검색은 공유해도 답변은 mode별 새 ID다.
ordered=sorted(retrievals,key=lambda r:(r['query_id'],(r['mode'] if int(r['query_id'][1:])%2 else ''.join(chr(255-ord(c)) for c in r['mode'])),r['profile']))
for index,row in enumerate(ordered):
 q=next(q for q in queries if q['query_id']==row['query_id'])
 base_u=next((r['result'] for r in retrievals if r['query_id']==row['query_id'] and r['profile']==row['profile'] and r['mode']=='U'),None)
 M.bind_retrieval(52,row,question=q,profile=row['profile'],mode=row['mode'],parser_record=by_query[q['query_id']],manifest=manifest,independent_u=base_u if row['mode']=='PAR' else None)
 answer=M.answer(52,row['identity'],q['question'],row['result'],allow_network=ALLOW_NETWORK)
 answers.append({**answer,**{k:row[k] for k in ['query_id','profile','mode','repeat']}})
 P.F.write_json(PATH/'progress.json',dict(phase='answer',completed=len(answers),expected=280,last_identity=row['identity'],ledger=X.ledger()))
 print('답변',row['identity'],answer['status'],round(answer['answer_ready_wall_seconds'],3),'초',flush=True)
P.F.write_json(PATH/'execution_results.json',dict(answers=answers,retrievals=retrievals,manifest_sha256=P.sha(manifest),resources=runtime.resources(),
 stage_wall_seconds=time.perf_counter()-started,stage_cpu_seconds=time.process_time()-cpu_started))
P.F.write_json(PATH/'completion_status.json',dict(status='awaiting_source_evaluation',logical_answers=len(answers),expected=280,stage53_allowed=False))
print('280조건 완료. 원문 평가·조건부 LLM 결합 진입 판정 전 최종 선정하지 않는다.',X.ledger())

In [ ]:
# 결과 원문 검토에서 순위 원인이 확인된 경우에만 추가 LLM 선택 비교를 실행한다.
from typing import Literal
from pydantic import create_model, Field, ValidationError

def select_existing_cards(q, profile, pool_result):
 identity=f"{q['query_id']}:{profile}:LLM_SELECTOR:r1";target=PATH/'selectors'/(identity+'.json')
 units=pool_result['trace']['ranked_units'];keys=sorted({u['card_key'] for u in units})
 if not keys:
  return dict(identity=identity,status='empty_pool',selection=[],events=[],wall_seconds=0.)
 schema=create_model('ExistingCardSelection'+str(len(keys)),cards=(list[Literal.__getitem__(tuple(keys))],Field(min_length=0,max_length=3)))
 content=P.F.canonical(dict(question=q['question'],candidate_units=[{k:u[k] for k in ['unit_id','card_key','card_name','origin','text']} for u in units]))
 request_input=[dict(role='user',content=content)]
 contract=dict(execution_manifest_sha256=P.sha(manifest),input_sha256_by_identity={identity:P.sha(request_input)},schema_sha256_by_identity={identity:P.sha(schema.model_json_schema())})
 X.frozen(PATH/'selector_contracts'/(identity+'.json'),contract)
 global_path=PATH/'answer_payload_contract.json';combined=P.F.read_json(global_path)
 for name in ['input_sha256_by_identity','schema_sha256_by_identity']:
  if identity in combined[name]:assert combined[name][identity]==contract[name][identity]
  combined[name].update(contract[name])
 P.F.write_json(global_path,combined)
 api=X.Cache(52,'answer',identity,allow_network=ALLOW_NETWORK)
 instructions='원질문과 후보 근거만 읽고 질문에 실제로 맞는 카드 ID를 최대3개 선택하세요. 후보 밖 카드는 만들지 마세요. 조건을 만족하는 카드가 없으면 빈 목록입니다. 추가 검색은 수행하지 않습니다.'
 if target.exists():row=P.F.read_json(target)
 else:
  start=time.perf_counter();selection=None;error=None
  for attempt in [1,2]:
   kwargs=dict(model='gpt-6-luna',instructions=instructions if attempt==1 else instructions+' 이전 출력의 형식 오류를 고쳐 완전한 JSON으로 답하세요.',input=request_input,text_format=schema,tools=[],store=False,max_output_tokens=2400)
   try:
    response=api.parse(**kwargs);selection=list(response.output_parsed.cards) if response.output_parsed else None
    if selection is None or len(set(selection))!=len(selection):raise ValueError('Empty or duplicate selector output')
    break
   except (ValidationError,ValueError) as exc:error=str(exc)
   except Exception as exc:
    if not api.events:raise
    error=f'{type(exc).__name__}: {exc}';break
  row=dict(identity=identity,status='ok' if selection is not None else 'failed',selection=selection or [],error=error,events=api.events,
   input_sha256=P.sha(request_input),schema_sha256=P.sha(schema.model_json_schema()),manifest_sha256=P.sha(manifest),fixed_pool_sha256=P.sha(pool_result),wall_seconds=time.perf_counter()-start)
  P.F.write_json(target,row)
 assert row['input_sha256']==P.sha(request_input) and row['schema_sha256']==P.sha(schema.model_json_schema()) and row['manifest_sha256']==P.sha(manifest) and row['fixed_pool_sha256']==P.sha(pool_result)
 assert set(row['selection'])<=set(keys)
 for event in row['events']:
  raw=P.F.read_json(PATH/'api_cache'/(event['key']+'.json'));request=raw['request']
  kwargs={k:v for k,v in request.items() if k!='schema'};kwargs['text_format']=schema
  assert api.key(kwargs)==event['key'] and P.sha(request['input'])==row['input_sha256'] and P.sha(request['schema'])==row['schema_sha256']
  assert request['model']=='gpt-6-luna' and request['max_output_tokens']==2400 and request['reasoning']=={'effort':'medium'}
 if row['status']=='ok':
  assert row['events'] and P.F.read_json(PATH/'api_cache'/(row['events'][-1]['key']+'.json'))['parsed']['cards']==row['selection']
 return row

gate_path=PATH/'conditional_trigger_review.json'
if not gate_path.exists():
 print('조건부 LLM 비교는 원문 평가의 진입 판정 대기 중이다. 추가 API 호출 없음.')
else:
 gate=P.F.read_json(gate_path)
 assert gate['manifest_sha256']==P.sha(manifest) and gate['execution_results_sha256']==P.sha(P.F.read_json(PATH/'execution_results.json'))
 review=P.F.read_json(PATH/'source_review.json')
 assert gate['source_review_sha256']==P.sha(review)
 assert review['execution_results_sha256']==gate['execution_results_sha256']
 assert gate['annotations_sha256']==review['annotations_sha256']==manifest['annotations_sha256']==P.sha(P.F.read_json(PATH/'frozen_annotations.json'))
 source_inputs={str((P.DATA/'30_structured_benefit_db'/name).relative_to(ROOT)):P.sha((P.DATA/'30_structured_benefit_db'/name).read_bytes()) for name in ['structured.sqlite','documents.jsonl','facts.jsonl']}
 source_inputs.update({'data/raw/'+d['document_id']+'.pdf':d['source_pdf_sha256'] for d in Q.documents()})
 assert review['source_input_sha256']==gate['source_input_sha256']==source_inputs
 for name,expected in source_inputs.items():assert P.sha((ROOT/name).read_bytes())==expected
 reviewed=review['query_profile_results']
 assert len(reviewed)==80 and {(r['query_id'],r['profile']) for r in reviewed}=={(q['query_id'],pf) for q in queries for pf in X.PROFILES}
 confirmed=set()
 for checked in reviewed:
  q=next(q for q in queries if q['query_id']==checked['query_id'])
  assert checked['question_sha256']==P.sha(q['question'])
  source=next(r for r in retrievals if r['query_id']==q['query_id'] and r['profile']==checked['profile'] and r['mode']=='PAR')
  pool=set(source['result']['trace']['candidate_cards']);top={c['card_key'] for c in source['result']['cards']}
  assert checked['PAR_result_sha256']==P.sha(source['result'])
  judgments={c['card_key']:c for c in checked['card_judgments']}
  assert top<=set(judgments)
  valid={k for k,c in judgments.items() if c['valid_for_original_question'] and k in pool}
  for key,c in judgments.items():
   assert c['reason'] and c['source_support']
   for support in c['source_support']:
    assert support['path']=='data/raw/'+key+'.pdf' and source_inputs[support['path']]==support['sha256'] and support['page']>=1 and support['observation']
  if checked['ranking_cause_confirmed']:
   assert checked['failure_cause']=='ranking' and valid and not (top&valid)
   assert all(not judgments[key]['valid_for_original_question'] for key in top)
   confirmed.add(checked['query_id'])
 assert set(gate['source_adjudicated_ranking_failure_query_ids'])==confirmed
 assert gate['trigger']==bool(confirmed)
 if not gate['trigger']:
  print('BGE 순위 원인 진입 조건 미충족: 조건부 LLM 호출 없음.')
 else:
  assert gate['source_adjudicated_ranking_failure_query_ids'] and gate['all_cases_evaluated']
  selectors=[];alternative=[]
  for q in queries:
   for pf in X.PROFILES:
    bge=next(r for r in retrievals if r['query_id']==q['query_id'] and r['profile']==pf and r['mode']=='PAR')
    u=next(r['result'] for r in retrievals if r['query_id']==q['query_id'] and r['profile']==pf and r['mode']=='U')
    M.bind_retrieval(52,bge,question=q,profile=pf,mode='PAR',parser_record=by_query[q['query_id']],manifest=manifest,independent_u=u)
    selector=select_existing_cards(q,pf,bge['result']);selectors.append(selector)
    kept=[unit for unit in bge['result']['trace']['ranked_units'] if unit['card_key'] in selector['selection']]
    result=M.pack_ranked(q['question'],kept,[unit['score'] for unit in kept])
    if bge['original_scope']['clarify']:result['clarify']=True
    M.assert_scope(result,bge['original_scope']['card_keys'])
    identity=f"{q['query_id']}:{pf}:PAR_LLM:r1"
    a=M.answer(52,identity,q['question'],result,allow_network=ALLOW_NETWORK)
    alternative.append(dict(answer=a,query_id=q['query_id'],profile=pf,mode='PAR_LLM',repeat=1,result=result,selector_sha256=P.sha(selector),selector_status=selector['status']))
    P.F.write_json(PATH/'conditional_progress.json',dict(completed=len(alternative),expected=80,ledger=X.ledger()))
  P.F.write_json(PATH/'conditional_execution_results.json',dict(selectors=selectors,answers=alternative,decision_pending_source_evaluation=True,
   card_choice='LLM subset of same fixed pool; BGE order retained among chosen cards/evidence',ledger=X.ledger()))
  print('조건부80문항·프로필 실행완료. BGE와 원문 품질 및 비용 비교 뒤 선정한다.')


### 실행 구간 기록

2026-10-22. 두 실행이 종료 코드 143(SIGTERM)으로 중단됐다. 정확한 외부 종료 원인은 확인되지 않았다. 확인한 최근 커널 로그에는 OOM 종료 기록이 없었지만 이것만으로 모든 메모리 문제를 배제하지는 않는다. 첫 중단 시 검색 결과 122개(H17 완료, H18 일부), 두 번째 중단 시 134개(H19 완료, H20 일부)가 저장되어 있었다. 답변 API 요청은 두 중단 시점 모두 0회였다. 질문 임베딩 1회 비용은 $0.000005080이며 중복 생성하지 않았다.

재개 시 동결 코드·입력·원래 임베딩 응답과 벡터·검색 결과 해시를 검증한다. 이후 임시 실행기는 약 150초가 지난 뒤 다음 진행 체크포인트 저장 직후 구간을 종료한다. 실험 로직과 동결된 노트북 코드 셀은 바꾸지 않는다. 임시 실행기는 보고서를 만들지 않는다.

구간별 자원은 해당 데이터 폴더의 `execution_segments/`에 저장한다. 개별 검색·실제 API 요청·논리 답변의 최초 측정 시간을 유지하며 재생 시간과 구분한다. 처음 두 강제 종료 구간 전체 CPU·최대 RSS는 미측정이다. 중복 사용한 검색 시간을 여러 번 합산하지 않으며, 실행 구간 시간이나 별도로 실행한 단계 합계를 온라인 종단 응답시간으로 부르지 않는다. 첫 토큰과 브라우저 표시 시간은 미측정이다.
